# Final results and error analysis

**Draft.** This notebook intentionally requires complete, validated reports rather than displaying placeholder metrics.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BASE = ROOT/'reports/baseline_reproduction/per_run.csv'
ROB = ROOT/'reports/lighting_blur/per_run.csv'
IMP = ROOT/'reports/reference_augmentation/per_run.csv'
for f in [BASE,ROB,IMP]:
    assert f.is_file(), f'Missing validated report: {f}'
base,rob,imp = [pd.read_csv(f) for f in [BASE,ROB,IMP]]
assert (len(base),len(rob),len(imp)) == (45,180,900)

## Reproduction and robustness

In [ ]:
metrics = ['image_auroc','pixel_aupro']
display((base.groupby('category')[metrics].mean()*100).round(2))
for m in metrics:
    delta=rob.groupby(['category','condition'])[m+'_change_pp'].mean().unstack()
    display(delta.round(2))
    ax=delta.plot.bar(figsize=(13,4)); ax.set_ylabel(m+' change (pp)'); plt.tight_layout(); plt.show()

## Proposed method: clean trade-off and robustness

In [ ]:
for m in metrics:
    grouped=imp.groupby(['category','variant','condition'])[m+'_delta_pp'].mean()
    macro=grouped.groupby(['variant','condition']).mean().unstack()
    display(macro.round(3))
    ax=macro.T.plot.bar(figsize=(10,4)); ax.set_ylabel(m+' paired change (pp)'); plt.tight_layout(); plt.show()
    worst=grouped.xs('combined',level='variant').sort_values().head(8)
    display(worst.to_frame('worst combined differences (pp)'))

## Interpretation checklist
- Inspect failure cases by category and condition, using aligned ground truth and anomaly maps.
- Separate archived baseline from same-pipeline comparator; report parity differences.
- Discuss localization vs classification, normal-image false positives, compute costs and negative results.
- Address zero-shot scope separately; do not invent WinCLIP results.
- Write final conclusions only after reviewing measured tables and figures.